# 05 — Anomaly detection

El dataset incluye `analytics_anomalias.parquet` con hallazgos del detector baseline (7 reglas en 3 severidades). Este notebook explora los resultados y muestra cómo usar el detector para análisis custom.

**Reglas**:
- CRITICAL: `validos_gt_emitidos`, `asistentes_gt_habiles`, `participacion_gt_100` (matemáticamente imposibles)
- HIGH: `suma_votos_mismatch`, `emitidos_gt_padron_reniec` (cross-check)
- MEDIUM: `concentracion_extrema`, `participacion_outlier_distrito` (estadísticos, no necesariamente ilegítimos)

In [ ]:
import polars as pl

anom = pl.read_parquet('parquet/analytics_anomalias.parquet')
print(f'total hallazgos: {anom.shape[0]:,}')
anom.group_by(['severidad', 'tipo']).agg(pl.len().alias('n')).sort(['severidad', 'n'], descending=[False, True])

In [ ]:
# Hallazgos por distrito electoral
anom.filter(pl.col('idDistritoElectoral').is_not_null()).group_by('idDistritoElectoral').agg(
    pl.len().alias('hallazgos')
).sort('hallazgos', descending=True).head(10)

In [ ]:
# Muestra de outliers de participación
participacion = anom.filter(pl.col('tipo') == 'participacion_outlier_distrito').sort(
    pl.col('valor_observado'), descending=True
).head(10)
participacion.select(['idActa', 'idEleccion', 'ubigeoDistrito', 'valor_observado', 'mensaje'])

In [ ]:
# Distritos con concentración extrema (1 partido >= 95%)
concentr = anom.filter(pl.col('tipo') == 'concentracion_extrema').sort(
    pl.col('valor_observado'), descending=True
)
concentr.select(['idActa', 'idEleccion', 'ubigeoDistrito', 'valor_observado', 'mensaje']).head(10)

In [ ]:
# Cross-check: las actas flaggeadas ¿son en zonas conocidas por baja competencia?
cab = pl.read_parquet('parquet/actas_cabecera.parquet').select([
    'idActa', 'ubigeoDistrito', 'nombreDistrito', 'idAmbitoGeografico'
])
concentr.join(cab, on='idActa', how='inner').group_by(['idAmbitoGeografico']).agg(
    pl.len().alias('concentracion_extrema')
)